# TN3K pipeline v2: feature extraction → XGBoost benchmark → LLM (Colab)

One notebook for the whole v2 pipeline:

- **Part A** re-extracts the radiomics features from the TN3K images with the corrected settings (report Section 11.6). The result is cached on Drive, so it runs once; later runs load it in seconds.
- **Part B** compares v1 and v2 features with XGBoost on the exchangeable split (the benchmark).
- **Part C** fine-tunes the LLM on the chosen features and evaluates it: classification, LTT, and the miss-rate guarantee. Results are added to the same `summary.csv` as the earlier sweep runs.

Earlier notebooks and v1 results are untouched: `TN3K_pipeline_end_to_end.ipynb`, `model_sweep_colab.ipynb`, `radiomics_v2_colab.ipynb` and `features/`.

**How to use**

1. Runtime → Change runtime type → **L4 GPU**. Parts A–B run on the CPU cores of the same machine.
2. Check the **Configuration** cell, then Runtime → **Run all**.
3. To try another LLM or setting: change the Configuration cell, Runtime → **Restart session**, then **Run all**. Part A is loaded from the cache.
4. When finished: Runtime → **Disconnect and delete runtime**.

## 1. Setup: repository, packages, Google Drive

In [1]:
import os
if not os.path.exists("/content/thesis"):
    !git clone -q https://github.com/sezermzgl/thesis.git /content/thesis
else:
    !git -C /content/thesis pull -q
%cd /content/thesis
!pip install -q -U bitsandbytes peft accelerate mrmr-selection xgboost SimpleITK

# PyRadiomics from PyPI fails to build on recent Python versions; install it from GitHub instead.
try:
    import radiomics
except ImportError:
    !pip install -q git+https://github.com/AIM-Harvard/pyradiomics.git
    import radiomics
print("PyRadiomics", radiomics.__version__)

from google.colab import drive
drive.mount("/content/drive")

/content/thesis
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.9/107.9 kB 12.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.9 MB/s eta 0:00:00
PyRadiomics 3.1.1.dev111+g8ed579383
Mounted at /content/drive


## 2. Configuration

**What to run.** `FEATURE_SET = "v2"` runs Part A (or loads its cache) and Part B, then trains the LLM on v2. `"v1"` skips A–B and trains on the repo's original features.

**LLM backbones**

Candidate backbones (all have a sequence-classification head and fit in 4-bit):

| `MODEL_NAME` | size | access | suggested `BATCH` / `GRAD_ACCUM` |
|---|---|---|---|
| `Qwen/Qwen2.5-1.5B` | 1.5B | open, the current model; **run first as the control** | 8 / 2 |
| `Qwen/Qwen2.5-3B` | 3B | open | 8 / 2 |
| `Qwen/Qwen2.5-7B` | 7B | open | 4 / 4 |
| `meta-llama/Llama-2-7b-hf` | 7B | gated; the model used by Ra et al. | 4 / 4 |
| `meta-llama/Llama-3.1-8B` | 8B | gated | 4 / 4 |
| `mistralai/Mistral-7B-v0.3` | 7B | may ask to accept terms | 4 / 4 |
| `Qwen/Qwen2.5-14B`, `Qwen/Qwen3-14B-Base`, `microsoft/phi-4` | 14B | open; L4 or A100 | 2 / 8 |

For a gated model, accept its license on the Hugging Face model page with your own account, then log in in a new cell (`from huggingface_hub import login; login()`).

`BATCH × GRAD_ACCUM` = 16 in every row, so the effective batch size is the same as in the main notebook.

In [2]:
# What to run
FEATURE_SET  = "v2"              # "v2": re-extracted features (Part A); "v1": repo features, Parts A-B skipped
SPLIT_SCHEME = "pooled_seed42"   # "pooled_seed42": exchangeable split; "official": main notebook's splits

# Part A: feature extraction (v2)
TN3K_ROOT              = "/content/drive/MyDrive/thesis_baseline/tn3k"
FEATURES_ROOT          = "/content/drive/MyDrive/thesis_baseline/tn3k_gt_radiomics_v2"
BIN_COUNT              = 64
FEATURE_FAMILIES       = ["shape2D", "firstorder", "glcm", "glszm", "glrlm", "gldm", "ngtdm"]
USE_LOG                = False          # LoG-filtered images (adds ~3x features); use a new FEATURES_ROOT
LOG_SIGMAS             = [1.0, 2.0, 3.0]
USE_WAVELET            = False          # wavelet sub-bands (adds ~4x features); use a new FEATURES_ROOT
NORMALIZE_TO_UINT8     = True           # per-image min-max to 0-255, as in v1
KEEP_LARGEST_COMPONENT = True
MIN_MASK_PIXELS        = 20
USE_CACHE              = True           # load Part A from FEATURES_ROOT if it exists
N_JOBS                 = -1

# Part C: LLM
MODEL_NAME   = "Qwen/Qwen2.5-1.5B"
N_FEATURES   = 16       # mRMR radiomics features in the prompt (0 with CLINICAL_FEATURES: TI-RADS features only)
MAX_CORR     = 0.95     # before mRMR, drop near-duplicate features (|r| above this, train only); None: off
CLINICAL_FEATURES = True   # TI-RADS-motivated features (conformal_triage/clinical.py): computed in Part A,
                            # compared with XGBoost in Part B, always in the prompt (first) in Part C.
                            # Needs FEATURE_SET = "v2"; MAX_LENGTH 512 (decimal) or 1536 (rich)
VALUE_FORMAT = "decimal"  # how values are written in the prompt:
                          #   "decimal":    "... is measured at -1.140."  (format of all earlier runs)
                          #   "zscore":     "... has a z-score of -1.14 relative to the training nodules."
                          #   "percentile": "... is at percentile 13 of the training nodules."
                          #   "rich":       z-score + percentile + band and meaning ("... lower than typical,
                          #                 meaning a less round, more irregular outline"); MAX_LENGTH 1024
                          #   "semantic_only": band and meaning only, no numbers; MAX_LENGTH 512
EPOCHS       = 5
LR           = 2e-4
LORA_TARGETS = "qv"     # "qv": attention Q/V projections (all earlier runs); "all": every linear layer, as in Ra et al. (2025)
BATCH        = 8        # per-device train batch size (7-8B: 4)
GRAD_ACCUM   = 2        # BATCH x GRAD_ACCUM = 16       (7-8B: 4)
MAX_LENGTH   = 512      # Part C stops before training if a prompt would be truncated
SEED         = 42
PRECISION    = "auto"   # "auto": bf16 on A100/L4, fp16 on T4
OUT_DIR      = "/content/drive/MyDrive/thesis_model_sweep"   # shared with the earlier sweep runs

# Risk levels, as in the main notebook (D6)
ALPHA_BENIGN, ALPHA_MALIGNANT, DELTA = 0.05, 0.20, 0.05

## 3. Imports, paths, GPU

In [3]:
import json, math, sys, time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
import torch.nn as nn
from IPython.display import display
from joblib import Parallel, delayed
import radiomics
from radiomics import featureextractor
from scipy import ndimage as ndi
from scipy.stats import binom
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

REPO = Path("/content/thesis")
sys.path.insert(0, str(REPO))
import conformal_triage as ct

torch.manual_seed(SEED); np.random.seed(SEED)
RUN_PART_A = FEATURE_SET == "v2"

TN3K_ROOT, FEATURES_ROOT = Path(TN3K_ROOT), Path(FEATURES_ROOT)
FEATURES_ROOT.mkdir(parents=True, exist_ok=True)
FEATURES_V2 = FEATURES_ROOT / "radiomics_v2_all.csv"
PIXEL_SHAPE = FEATURES_ROOT / "radiomics_v2_pixel_shape.csv"
ERRORS_V2   = FEATURES_ROOT / "radiomics_v2_errors.csv"
V2_FEATURES_CSV = str(FEATURES_V2)
CLINICAL_CSV = FEATURES_ROOT / "clinical_features.csv"
MERGED_CSV   = FEATURES_ROOT / "radiomics_v2_clinical.csv"
CLIN = ct.clinical.COLUMNS
assert not (CLINICAL_FEATURES and FEATURE_SET != "v2"), "CLINICAL_FEATURES needs FEATURE_SET = 'v2'"

GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
if PRECISION == "auto":
    # bf16 needs compute capability >= 8 (A100, L4). Checked directly, because
    # torch.cuda.is_bf16_supported() can report emulated (slow) bf16 on a T4.
    PRECISION_USED = "bf16" if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else "fp16"
else:
    PRECISION_USED = PRECISION
DTYPE = torch.bfloat16 if PRECISION_USED == "bf16" else torch.float16

RUN_TAG = f"{MODEL_NAME.split('/')[-1]}_f{N_FEATURES}_e{EPOCHS}_s{SEED}"
if SPLIT_SCHEME != "official":
    RUN_TAG += f"_{SPLIT_SCHEME}"
if FEATURE_SET != "v1":
    RUN_TAG += f"_{FEATURE_SET}"
if VALUE_FORMAT != "decimal":
    RUN_TAG += f"_{VALUE_FORMAT}"
if MAX_CORR is not None:
    RUN_TAG += f"_corr{round(MAX_CORR * 100)}"
if CLINICAL_FEATURES:
    RUN_TAG += f"_clin{len(CLIN)}"     # number of TI-RADS features, so runs with a newer feature set get a new folder
if LR != 2e-4:
    RUN_TAG += "_lr" + f"{LR:.0e}".replace("e-0", "e-")
if LORA_TARGETS != "qv":
    RUN_TAG += f"_lora-{LORA_TARGETS}"
RUN_DIR = Path(OUT_DIR) / RUN_TAG
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"features: {FEATURE_SET} | split: {SPLIT_SCHEME} | GPU: {GPU} | precision: {PRECISION_USED} | run: {RUN_TAG}")
if RUN_PART_A:
    for p in ["trainval-image", "trainval-mask", "test-image", "test-mask", "label4trainval.csv", "label4test.csv"]:
        print(f"{'OK' if (TN3K_ROOT / p).exists() else 'MISSING':8} {TN3K_ROOT / p}")

features: v2 | split: pooled_seed42 | GPU: NVIDIA L4 | precision: bf16 | run: Qwen2.5-1.5B_f16_e5_s42_pooled_seed42_v2_corr95_clin7
OK       /content/drive/MyDrive/thesis_baseline/tn3k/trainval-image
OK       /content/drive/MyDrive/thesis_baseline/tn3k/trainval-mask
OK       /content/drive/MyDrive/thesis_baseline/tn3k/test-image
OK       /content/drive/MyDrive/thesis_baseline/tn3k/test-mask
OK       /content/drive/MyDrive/thesis_baseline/tn3k/label4trainval.csv
OK       /content/drive/MyDrive/thesis_baseline/tn3k/label4test.csv


# Part A: radiomics v2 extraction

**Nothing from v1 is changed or overwritten.** v1 stays in `features/radiomics_*.csv` in the repo. v2 is written to its own Drive folder (`OUT_ROOT`).

| setting | v1 (main notebook) | v2 (this notebook) |
|---|---|---|
| grey levels for texture | `binWidth = 25` (about 9 levels inside a nodule) | `binCount = 64` |
| shape features | pixels; image sizes differ across images | size features divided by the image scale (`…Relative`); pixel values kept in a separate diagnostics file |
| feature families | shape2D, first-order, GLCM, GLSZM | + GLRLM, GLDM, NGTDM |
| filtered images | none | optional LoG / wavelet (off by default) |
| multi-component masks | used as they are | only the largest component is kept |
| border-touching masks | used as they are | kept and flagged (`touches_border`) |
| intensity normalisation | per-image min–max to 0–255 | unchanged, so the comparison isolates the planned changes |

## A1. Manifest: images, masks and labels

Same pairing as main notebook A3. `sample_id` (`trainval/<n>` or `test/<n>`) matches `splits/pooled_seed42.csv` and the v1 tables.

In [4]:
if RUN_PART_A:
    LABEL_NAMES = {0: "benign", 1: "malignant"}
    IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

    def path_map(folder):
        return {p.stem: p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXT}

    rows = []
    for official, labels_file in [("trainval", "label4trainval.csv"), ("test", "label4test.csv")]:
        labels = pd.read_csv(TN3K_ROOT / labels_file, header=None, names=["image_name", "label"], dtype={"image_name": str})
        labels["id"] = labels["image_name"].str.strip().map(lambda x: Path(x).stem)
        images, masks = path_map(TN3K_ROOT / f"{official}-image"), path_map(TN3K_ROOT / f"{official}-mask")
        assert set(images) == set(masks) == set(labels["id"]), f"{official}: images, masks and labels do not match"
        for _, r in labels.iterrows():
            rows.append({"id": r["id"], "sample_id": f"{official}/{int(r['id'])}", "image_name": images[r["id"]].name,
                         "official_split": official, "label": int(r["label"]), "label_name": LABEL_NAMES[int(r["label"])],
                         "image_path": str(images[r["id"]]), "mask_path": str(masks[r["id"]])})
    manifest = pd.DataFrame(rows)
    assert manifest["sample_id"].is_unique

    split_ids = set(pd.read_csv(REPO / "splits" / "pooled_seed42.csv")["sample_id"])
    print("images:", len(manifest), "| all in splits/pooled_seed42.csv:", set(manifest["sample_id"]) == split_ids)
    display(manifest.groupby(["official_split", "label_name"]).size().rename("n").to_frame())
else:
    print('skipped (FEATURE_SET = "v1")')

images: 3493 | all in splits/pooled_seed42.csv: True


n
official_split label_name      
test           benign       378
               malignant    236
trainval       benign      1905
               malignant    974

## A2. Extraction function

- Mask: binarised at 127; if it has several connected components, only the largest is kept (recorded in `n_components_raw` and `kept_component_fraction`).
- Image: grey-scale, per-image min–max to 0–255 (as v1), pixel spacing 1 × 1 (no physical spacing in the JPEGs).
- Size features are divided by the image scale `s = sqrt(width × height)`: lengths by `s`, areas by `s²`, and the perimeter/area ratio is multiplied by `s`. They are stored as `…Relative`. The pixel values go to a separate diagnostics file and are not used as features. Elongation and sphericity do not depend on scale and are kept as they are.

In [5]:
if RUN_PART_A:
    LENGTHS = ["MajorAxisLength", "MinorAxisLength", "MaximumDiameter", "Perimeter"]
    AREAS   = ["MeshSurface", "PixelSurface"]
    INVERSE = ["PerimeterSurfaceRatio"]

    def make_extractor():
        ex = featureextractor.RadiomicsFeatureExtractor()
        ex.disableAllImageTypes()
        ex.enableImageTypeByName("Original")
        if USE_LOG:
            ex.enableImageTypeByName("LoG", customArgs={"sigma": LOG_SIGMAS})
        if USE_WAVELET:
            ex.enableImageTypeByName("Wavelet")
        ex.disableAllFeatures()
        for family in FEATURE_FAMILIES:
            ex.enableFeatureClassByName(family)
        ex.settings.update(force2D=True, force2Ddimension=0, binCount=BIN_COUNT,
                           resampledPixelSpacing=None, correctMask=True)
        return ex

    def prepare_mask(mask_path):
        raw = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        mask = raw > 127
        labelled, n = ndi.label(mask)
        kept = 1.0
        if n > 1 and KEEP_LARGEST_COMPONENT:
            sizes = ndi.sum(mask, labelled, index=range(1, n + 1))
            largest = int(np.argmax(sizes)) + 1
            kept = float(sizes.max() / mask.sum())
            mask = labelled == largest
        touches = bool(mask[0, :].any() or mask[-1, :].any() or mask[:, 0].any() or mask[:, -1].any())
        return mask.astype(np.uint8), {"n_components_raw": int(n), "kept_component_fraction": kept,
                                       "touches_border": touches, "mask_pixels": int(mask.sum())}

    def extract_one(row):
        try:
            gray = cv2.imread(row["image_path"], cv2.IMREAD_GRAYSCALE).astype(np.float32)
            mask, mask_info = prepare_mask(row["mask_path"])
            if gray.shape != mask.shape:
                raise ValueError(f"image/mask shape mismatch {gray.shape} vs {mask.shape}")
            if mask_info["mask_pixels"] < MIN_MASK_PIXELS:
                raise ValueError(f"mask has {mask_info['mask_pixels']} pixels")
            if NORMALIZE_TO_UINT8:
                lo, hi = float(gray.min()), float(gray.max())
                gray = np.zeros_like(gray) if hi <= lo else np.round((gray - lo) / (hi - lo) * 255)
            img, msk = sitk.GetImageFromArray(gray), sitk.GetImageFromArray(mask)
            img.SetSpacing((1.0, 1.0)); msk.SetSpacing((1.0, 1.0))
            feats = {k: float(v) for k, v in make_extractor().execute(img, msk).items() if not k.startswith("diagnostics")}

            h, w = gray.shape
            scale = math.sqrt(h * w)
            pixel_shape = {}
            for key in list(feats):
                image_type, family, name = key.split("_", 2)
                if family != "shape2D":
                    continue
                if name in LENGTHS + AREAS + INVERSE:
                    pixel_shape[key] = feats.pop(key)
                    factor = scale if name in LENGTHS else scale**2 if name in AREAS else 1 / scale
                    feats[f"{image_type}_shape2D_{name}Relative"] = pixel_shape[key] / factor
            meta = {k: row[k] for k in ["id", "sample_id", "image_name", "official_split", "label", "label_name"]}
            return {**meta, "image_width": w, "image_height": h, **mask_info}, feats, pixel_shape, None
        except Exception as e:
            return None, None, None, {"sample_id": row["sample_id"], "error": f"{type(e).__name__}: {e}"}

    # Quick check on one image before the full run
    meta0, feats0, pix0, err0 = extract_one(manifest.iloc[0])
    assert err0 is None, err0
    print(f"one image: {len(feats0)} features | shape features: {sorted(k.split('_', 2)[2] for k in feats0 if '_shape2D_' in k)}")
else:
    print('skipped (FEATURE_SET = "v1")')

INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
INFO:radiomics.featureextractor:Computing shape2D
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INFO:radiomics.featureextractor:Computing firs

one image: 102 features | shape features: ['Elongation', 'MajorAxisLengthRelative', 'MaximumDiameterRelative', 'MeshSurfaceRelative', 'MinorAxisLengthRelative', 'PerimeterRelative', 'PerimeterSurfaceRatioRelative', 'PixelSurfaceRelative', 'Sphericity']


## A3. Extract all images (parallel, cached)

Running time depends on the CPU and on whether LoG or wavelet are enabled. Original-only should take minutes, not hours.

In [6]:
if RUN_PART_A:
    if USE_CACHE and FEATURES_V2.exists():
        v2 = pd.read_csv(FEATURES_V2)
        print("cached:", FEATURES_V2, v2.shape)
    else:
        start = time.time()
        results = Parallel(n_jobs=N_JOBS, verbose=5)(delayed(extract_one)(r) for _, r in manifest.iterrows())
        ok = [r for r in results if r[3] is None]
        errors = pd.DataFrame([r[3] for r in results if r[3] is not None])
        v2 = pd.DataFrame([{**m, **f} for m, f, _, _ in ok])
        pix = pd.DataFrame([{"sample_id": m["sample_id"], **p} for m, _, p, _ in ok])
        v2.to_csv(FEATURES_V2, index=False); pix.to_csv(PIXEL_SHAPE, index=False); errors.to_csv(ERRORS_V2, index=False)
        print(f"extracted {len(v2)} images in {(time.time() - start) / 60:.1f} min; errors: {len(errors)}")
        if len(errors):
            display(errors.head())

    (FEATURES_ROOT / "radiomics_v2_config.json").write_text(json.dumps({
        "bin_count": BIN_COUNT, "families": FEATURE_FAMILIES, "use_log": USE_LOG, "log_sigmas": LOG_SIGMAS,
        "use_wavelet": USE_WAVELET, "normalize_to_uint8": NORMALIZE_TO_UINT8,
        "keep_largest_component": KEEP_LARGEST_COMPONENT, "min_mask_pixels": MIN_MASK_PIXELS,
        "size_normalisation": "lengths / sqrt(W*H), areas / (W*H), perimeter-surface ratio * sqrt(W*H)",
        "pyradiomics": radiomics.__version__}, indent=2))

    META_V2 = ["id", "sample_id", "image_name", "official_split", "label", "label_name", "image_width",
               "image_height", "n_components_raw", "kept_component_fraction", "touches_border", "mask_pixels"]
    FEATS_V2 = [c for c in v2.columns if c not in META_V2]
    print(f"v2: {len(v2)} images, {len(FEATS_V2)} features, missing values: {int(v2[FEATS_V2].isna().sum().sum())}")
    print("multi-component masks:", int((v2["n_components_raw"] > 1).sum()), "| border-touching:", int(v2["touches_border"].sum()))
else:
    print('skipped (FEATURE_SET = "v1")')

cached: /content/drive/MyDrive/thesis_baseline/tn3k_gt_radiomics_v2/radiomics_v2_all.csv (3493, 114)
v2: 3493 images, 102 features, missing values: 0
multi-component masks: 298 | border-touching: 341


## A4. TI-RADS-motivated features (optional, `CLINICAL_FEATURES = True`)

PyRadiomics measures the nodule on its own. These features follow TI-RADS signs that depend on orientation or on the surroundings (`conformal_triage/clinical.py`): taller-than-wide ratio (shape), brightness against a ring of surrounding tissue (echogenicity), small bright spots (echogenic foci), solidity and border sharpness (margin), and the share of fluid-like dark pixels (composition). They are computed on the image as stored, with fixed settings, and cached on Drive. The table at the end shows each feature's AUC on the train split alone, as a sanity check.

In [7]:
if RUN_PART_A and CLINICAL_FEATURES:
    def clinical_one(row):
        try:
            gray = cv2.imread(row["image_path"], cv2.IMREAD_GRAYSCALE).astype(np.float64)
            mask, _ = prepare_mask(row["mask_path"])
            return {"sample_id": row["sample_id"], **ct.clinical_features(gray, mask.astype(bool))}
        except Exception as e:
            return {"sample_id": row["sample_id"], "error": f"{type(e).__name__}: {e}"}

    cached = pd.read_csv(CLINICAL_CSV) if USE_CACHE and CLINICAL_CSV.exists() else None
    if cached is not None and set(CLIN) <= set(cached.columns):
        clin = cached
        print("cached:", CLINICAL_CSV, clin.shape)
    else:   # no cache, or the cache predates a feature: recompute
        start = time.time()
        clin = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=0)(delayed(clinical_one)(r) for _, r in manifest.iterrows()))
        clin.to_csv(CLINICAL_CSV, index=False)
        print(f"computed for {len(clin)} images in {(time.time() - start) / 60:.1f} min")
    if "error" in clin.columns and clin["error"].notna().any():
        print("errors:", int(clin["error"].notna().sum())); display(clin[clin["error"].notna()].head())
    display(clin[CLIN].describe().T.round(3))

    merged = v2.merge(clin[["sample_id", *CLIN]], on="sample_id", how="left")
    merged.to_csv(MERGED_CSV, index=False)
    V2_FEATURES_CSV = str(MERGED_CSV)     # Part C reads radiomics + clinical features

    split_of = pd.read_csv(REPO / "splits" / "pooled_seed42.csv").set_index("sample_id")["split"]
    tr = merged[merged["sample_id"].map(split_of) == "train"]
    print("AUC of each feature on the train split (above 0.5: higher in malignant nodules):")
    display(pd.Series({c: roc_auc_score(tr["label"], tr[c].fillna(tr[c].median())) for c in CLIN}).round(3).to_frame("train AUC"))
else:
    print("skipped (CLINICAL_FEATURES = False)" if RUN_PART_A else 'skipped (FEATURE_SET = "v1")')

computed for 3493 images in 6.9 min


,count,mean,std,min,25%,50%,75%,max
original_clinical_TallerThanWideRatio,3493.0,0.760,0.223,0.259,0.606,0.730,0.874,2.237
original_clinical_EchogenicityRatio,3493.0,0.694,0.255,0.043,0.525,0.706,0.850,2.594
original_clinical_SolidEchogenicityRatio,3493.0,0.791,0.188,0.340,0.658,0.767,0.895,2.597
original_clinical_PunctateFociDensity,3493.0,2.026,2.101,0.000,0.605,1.394,2.729,16.598
original_clinical_Solidity,3493.0,0.985,0.029,0.705,0.985,0.999,1.000,1.000
original_clinical_MarginSharpness,3493.0,0.228,0.083,0.048,0.168,0.216,0.275,0.792
original_clinical_AnechoicFraction,3493.0,0.162,0.225,0.000,0.001,0.042,0.250,0.948


AUC of each feature on the train split (above 0.5: higher in malignant nodules):


,train AUC
original_clinical_TallerThanWideRatio,0.658
original_clinical_EchogenicityRatio,0.552
original_clinical_SolidEchogenicityRatio,0.509
original_clinical_PunctateFociDensity,0.487
original_clinical_Solidity,0.367
original_clinical_MarginSharpness,0.563
original_clinical_AnechoicFraction,0.409


## A5. Sanity check against v1

First-order features that do not depend on binning (e.g. Mean, Median, Maximum) should equal the v1 values on single-component masks. Any mismatch would point to a pipeline difference.

In [8]:
if RUN_PART_A:
    v1 = pd.concat([pd.read_csv(REPO / "features" / f"radiomics_{s}.csv") for s in ["train", "validation", "calibration", "test"]], ignore_index=True)
    v1["sample_id"] = v1["official_split"].astype(str) + "/" + v1["id"].astype(str)
    both = v2.merge(v1, on="sample_id", suffixes=("_v2", "_v1"))
    single = both[both["n_components_raw"] == 1]
    check = {}
    for name in ["original_firstorder_Mean", "original_firstorder_Median", "original_firstorder_Maximum", "original_shape2D_Elongation"]:
        a, b = single[f"{name}_v2"], single[f"{name}_v1"]
        check[name] = {"max abs difference": float((a - b).abs().max()), "correlation": float(np.corrcoef(a, b)[0, 1])}
    print(f"{len(both)} images matched to v1; {len(single)} with a single-component mask")
    display(pd.DataFrame(check).T)
else:
    print('skipped (FEATURE_SET = "v1")')

3493 images matched to v1; 3195 with a single-component mask


,max abs difference,correlation
original_firstorder_Mean,3.552714e-15,1.0
original_firstorder_Median,0.000000e+00,1.0
original_firstorder_Maximum,0.000000e+00,1.0
original_shape2D_Elongation,0.000000e+00,1.0


## A6. Does the image-size artefact shrink?

A classifier tries to tell official-test images from trainval images using the features. With v1, the official test set was easy to separate, largely through pixel-unit size features. If the size normalisation works, separation with the v2 shape features should drop.

In [9]:
if RUN_PART_A:
    def standardise(df, cols):
        x = df[cols].replace([np.inf, -np.inf], np.nan)
        return ((x - x.mean()) / x.std()).fillna(0).to_numpy()

    rows = {}
    v1_feats = [c for c in v1.columns if c.startswith("original_")]
    for name, table, cols in [
            ("v1, shape features (pixels)", v1, [c for c in v1_feats if "_shape2D_" in c]),
            ("v2, shape features (relative)", v2, [c for c in FEATS_V2 if "_shape2D_" in c]),
            ("v1, all features", v1, v1_feats),
            ("v2, all features", v2, FEATS_V2)]:
        X = standardise(table, cols)
        is_test = (table["official_split"] == "test").to_numpy()
        rows[name] = ct.two_sample_test(X[~is_test], X[is_test], seed=SEED)
    display(pd.DataFrame(rows).T.round(3))
else:
    print('skipped (FEATURE_SET = "v1")')

,n_a,n_b,auc,p_value
"v1, shape features (pixels)",2879.0,614.0,0.654,0.002
"v2, shape features (relative)",2879.0,614.0,0.709,0.002
"v1, all features",2879.0,614.0,0.708,0.002
"v2, all features",2879.0,614.0,0.805,0.002


# Part B: XGBoost benchmark, v1 vs v2

Same exchangeable split (`splits/pooled_seed42.csv`), same model (XGBoost with fixed settings, not tuned) and same mRMR procedure on train only; only the features differ. The last columns show how clean the model's most confident benign group is. Under LTT, the 5% target needs that group to be both large and very clean (report Section 11.5).

With `MAX_CORR` set, each feature set is also run after dropping near-duplicate features (column `corr filter`), as in Part C.

In [10]:
if RUN_PART_A:
    from mrmr import mrmr_classif
    from scipy.stats import binom
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import roc_auc_score
    from sklearn.preprocessing import StandardScaler
    from xgboost import XGBClassifier

    SPLITS = ["train", "validation", "calibration", "test"]
    assignment = pd.read_csv(REPO / "splits" / "pooled_seed42.csv").set_index("sample_id")["split"]

    def evaluate_features(table, feature_cols, label, max_corr=None):
        t = table.assign(split=table["sample_id"].map(assignment))
        assert t["split"].notna().all()
        parts = {s: t[t["split"] == s] for s in SPLITS}
        cols = [c for c in feature_cols if parts["train"][c].nunique() > 1]
        imp, sc = SimpleImputer(strategy="median"), StandardScaler()
        X = {"train": pd.DataFrame(sc.fit_transform(imp.fit_transform(parts["train"][cols].replace([np.inf, -np.inf], np.nan))), columns=cols)}
        for s in SPLITS[1:]:
            X[s] = pd.DataFrame(sc.transform(imp.transform(parts[s][cols].replace([np.inf, -np.inf], np.nan))), columns=cols)
        y = {s: parts[s]["label"].to_numpy() for s in SPLITS}
        if max_corr is not None:   # drop near-duplicate features (train only), as in Part C
            cols, _ = ct.drop_correlated(X["train"], y["train"], max_corr)
            X = {s: X[s][cols] for s in SPLITS}
        order = mrmr_classif(X=X["train"], y=pd.Series(y["train"]), K=len(cols), show_progress=False)
        out = []
        for k in [8, 16, 32, len(cols)]:
            f = order[:k]
            m = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                              scale_pos_weight=(y["train"] == 0).sum() / (y["train"] == 1).sum(),
                              random_state=SEED, verbosity=0).fit(X["train"][f], y["train"])
            p = {s: m.predict_proba(X[s][f])[:, 1] for s in SPLITS[1:]}
            lowest = np.argsort(p["calibration"])[: int(0.15 * len(p["calibration"]))]
            n_low, k_low = len(lowest), int(y["calibration"][lowest].sum())
            out.append({"features": label, "corr filter": "off" if max_corr is None else f"|r| > {max_corr}",
                        "n features": k,
                        **{f"AUC {s[:3]}": round(roc_auc_score(y[s], p[s]), 3) for s in SPLITS[1:]},
                        "lowest 15% of cal: cancers": f"{k_low}/{n_low} ({k_low / n_low:.1%})",
                        "5% certifiable on that group": bool(binom.cdf(k_low, n_low, 0.05) <= 0.05)})
        return pd.DataFrame(out), order

    v1_cols = [c for c in v1.columns if c.startswith("original_")]
    results = []
    for max_corr in ([None] if MAX_CORR is None else [None, MAX_CORR]):
        res_v1, order_v1 = evaluate_features(v1, v1_cols, "v1", max_corr)
        res_v2, order_v2 = evaluate_features(v2, FEATS_V2, "v2", max_corr)
        results += [res_v1, res_v2]
    comparison = pd.concat(results, ignore_index=True)
    display(comparison)
    comparison.to_csv(FEATURES_ROOT / "v1_vs_v2_xgboost_pooled.csv", index=False)
    print(f"mRMR top 16 (v2, MAX_CORR = {MAX_CORR}):")
    for i, f in enumerate(order_v2[:16], 1):
        print(f"  {i:2d}. {f}")
else:
    print('skipped (FEATURE_SET = "v1")')

,features,corr filter,n features,AUC val,AUC cal,AUC tes,lowest 15% of cal: cancers,5% certifiable on that group
0,v1,off,8,0.736,0.751,0.695,5/78 (6.4%),False
1,v1,off,16,0.758,0.769,0.723,8/78 (10.3%),False
2,v1,off,32,0.768,0.768,0.738,5/78 (6.4%),False
3,v1,off,67,0.798,0.825,0.793,4/78 (5.1%),False
4,v2,off,8,0.763,0.731,0.723,11/78 (14.1%),False
5,v2,off,16,0.786,0.812,0.780,4/78 (5.1%),False
6,v2,off,32,0.801,0.823,0.803,4/78 (5.1%),False
7,v2,off,102,0.798,0.819,0.800,2/78 (2.6%),False
8,v1,|r| > 0.95,8,0.740,0.749,0.691,9/78 (11.5%),False
9,v1,|r| > 0.95,16,0.749,0.763,0.717,7/78 (9.0%),False


mRMR top 16 (v2, MAX_CORR = 0.95):
   1. original_shape2D_MajorAxisLengthRelative
   2. original_glszm_SmallAreaLowGrayLevelEmphasis
   3. original_shape2D_Elongation
   4. original_shape2D_PixelSurfaceRelative
   5. original_firstorder_Range
   6. original_shape2D_PerimeterSurfaceRatioRelative
   7. original_gldm_DependenceVariance
   8. original_firstorder_MeanAbsoluteDeviation
   9. original_glcm_Idm
  10. original_shape2D_Sphericity
  11. original_gldm_DependenceNonUniformityNormalized
  12. original_gldm_DependenceNonUniformity
  13. original_glcm_MaximumProbability
  14. original_glszm_SizeZoneNonUniformity
  15. original_glszm_ZoneEntropy
  16. original_glszm_SizeZoneNonUniformityNormalized


## B1. Do the TI-RADS features add to XGBoost? (`CLINICAL_FEATURES = True`)

Same split, preprocessing and XGBoost settings as above. The radiomics features go through the near-duplicate filter and mRMR; the TI-RADS features are added on top, as in the Part C prompt.

In [11]:
if RUN_PART_A and CLINICAL_FEATURES:
    t = merged.assign(split=merged["sample_id"].map(assignment))
    parts = {s: t[t["split"] == s] for s in SPLITS}
    rad = [c for c in FEATS_V2 if parts["train"][c].nunique() > 1]
    cols = rad + CLIN
    imp, sc = SimpleImputer(strategy="median"), StandardScaler()
    clean = lambda df: df[cols].replace([np.inf, -np.inf], np.nan)
    X = {"train": pd.DataFrame(sc.fit_transform(imp.fit_transform(clean(parts["train"]))), columns=cols)}
    for s in SPLITS[1:]:
        X[s] = pd.DataFrame(sc.transform(imp.transform(clean(parts[s]))), columns=cols)
    y = {s: parts[s]["label"].to_numpy() for s in SPLITS}
    cand = rad if MAX_CORR is None else ct.drop_correlated(X["train"][rad], y["train"], MAX_CORR)[0]
    k = N_FEATURES if N_FEATURES > 0 else 16          # N_FEATURES = 0 (TI-RADS-only prompt): still show mRMR 16 for reference
    top = list(mrmr_classif(X=X["train"][cand], y=pd.Series(y["train"]), K=k, show_progress=False))
    settings = {f"radiomics, mRMR {k}": top, f"radiomics, mRMR {k} + TI-RADS": top + CLIN,
                "TI-RADS only": CLIN, f"radiomics, all {len(cand)}": cand, f"radiomics, all {len(cand)} + TI-RADS": cand + CLIN}
    rows = []
    for name, f in settings.items():
        m = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                          scale_pos_weight=(y["train"] == 0).sum() / (y["train"] == 1).sum(),
                          random_state=SEED, verbosity=0).fit(X["train"][f], y["train"])
        p = {s: m.predict_proba(X[s][f])[:, 1] for s in SPLITS[1:]}
        lowest = np.argsort(p["calibration"])[: int(0.15 * len(p["calibration"]))]
        k_low = int(y["calibration"][lowest].sum())
        rows.append({"features": name, "n": len(f), **{f"AUC {s[:3]}": round(roc_auc_score(y[s], p[s]), 3) for s in SPLITS[1:]},
                     "lowest 15% of cal: cancers": f"{k_low}/{len(lowest)}"})
    clinical_xgb = pd.DataFrame(rows)
    display(clinical_xgb)
    clinical_xgb.to_csv(FEATURES_ROOT / "xgboost_clinical_pooled.csv", index=False)
else:
    print("skipped")

,features,n,AUC val,AUC cal,AUC tes,lowest 15% of cal: cancers
0,"radiomics, mRMR 16",16,0.799,0.823,0.805,4/78
1,"radiomics, mRMR 16 + TI-RADS",23,0.828,0.847,0.829,3/78
2,TI-RADS only,7,0.783,0.791,0.774,9/78
3,"radiomics, all 54",54,0.805,0.816,0.796,2/78
4,"radiomics, all 54 + TI-RADS",61,0.821,0.838,0.826,3/78


# Part C: LLM fine-tuning and evaluation

## C1. Data: cached radiomics tables and splits

With `SPLIT_SCHEME = "pooled_seed42"`, all 3,493 images (official trainval + official test) are pooled and re-split by `splits/pooled_seed42.csv` (made by `experiments/make_pooled_splits.py`). The split is stratified by label and by official source, so calibration and test come from the same distribution and the conformal guarantees apply to the test split (report Section 5). With `"official"`, the main notebook's splits are used.

`FEATURE_SET = "v1"` uses the repo's radiomics tables (main notebook). `"v2"` uses the re-extracted features from `experiments/radiomics_v2_colab.ipynb`, read from Drive. Its extra metadata columns (image size, mask flags) are not used as features.

In [12]:
SPLITS = ["train", "validation", "calibration", "test"]
v1_tables = {s: pd.read_csv(REPO / "features" / f"radiomics_{s}.csv") for s in SPLITS}
pool = (pd.concat(v1_tables.values(), ignore_index=True) if FEATURE_SET == "v1"
        else pd.read_csv(V2_FEATURES_CSV))
pool["sample_id"] = pool["official_split"].astype(str) + "/" + pool["id"].astype(int).astype(str)
if SPLIT_SCHEME == "official":
    assignment = pd.concat([t.assign(split=s) for s, t in v1_tables.items()])
    assignment = assignment.set_index(assignment["official_split"].astype(str) + "/" + assignment["id"].astype(str))["split"]
else:
    assignment = pd.read_csv(REPO / "splits" / f"{SPLIT_SCHEME}.csv").set_index("sample_id")["split"]
pool["new_split"] = pool["sample_id"].map(assignment)
assert pool["new_split"].notna().all(), "some images are missing from the split assignment"
tables = {s: pool[pool["new_split"] == s].drop(columns="new_split").reset_index(drop=True) for s in SPLITS}

print("Feature set:", FEATURE_SET)
print("Split scheme:", SPLIT_SCHEME)
display(pd.DataFrame({s: {"nodules": len(t), "malignant": int(t["label"].sum()),
                          "malignant share": round(t["label"].mean(), 3),
                          "from official test": round((t["official_split"] == "test").mean(), 3)}
                      for s, t in tables.items()}).T)

Feature set: v2
Split scheme: pooled_seed42


,nodules,malignant,malignant share,from official test
train,2095.0,725.0,0.346,0.176
validation,436.0,151.0,0.346,0.174
calibration,524.0,182.0,0.347,0.177
test,438.0,152.0,0.347,0.176


## C2. Train-only standardization and mRMR feature selection

Same as main notebook A10: median imputation and z-scoring are fitted on train only, and mRMR picks `N_FEATURES` features on train.

With `MAX_CORR` set, near-duplicate features are removed first (train only): of two features with |r| above `MAX_CORR`, the one less related to the label is dropped. mRMR alone does not prevent this; on v2 it picked both `PixelSurfaceRelative` and `MeshSurfaceRelative`, which are the same number in 2D.

In [13]:
from mrmr import mrmr_classif

META_COLUMNS = ["id", "sample_id", "image_name", "official_split", "analysis_split",
                "label", "label_name", "mask_source", "source_split",
                "image_width", "image_height", "n_components_raw", "kept_component_fraction", "touches_border", "mask_pixels"]   # v2 metadata, not features
train = tables["train"]
feature_columns = [c for c in train.columns
                   if c not in META_COLUMNS and pd.api.types.is_numeric_dtype(train[c])]
raw = {s: t.reindex(columns=feature_columns).replace([np.inf, -np.inf], np.nan) for s, t in tables.items()}
valid = [c for c in feature_columns
         if raw["train"][c].notna().sum() > 0 and raw["train"][c].nunique(dropna=True) > 1]

imputer, scaler = SimpleImputer(strategy="median"), StandardScaler()
scaled = {"train": pd.DataFrame(scaler.fit_transform(imputer.fit_transform(raw["train"][valid])), columns=valid)}
for s in SPLITS[1:]:
    scaled[s] = pd.DataFrame(scaler.transform(imputer.transform(raw[s][valid])), columns=valid)

clinical = [c for c in valid if c in CLIN] if CLINICAL_FEATURES else []
radiomic = [c for c in valid if c not in CLIN]      # TI-RADS features never compete in the filter or mRMR
candidates = radiomic
if MAX_CORR is not None:
    candidates, duplicates = ct.drop_correlated(scaled["train"][radiomic], train["label"].astype(int), MAX_CORR)
    print(f"Near-duplicates dropped (|r| > {MAX_CORR} on train): {len(duplicates)} of {len(valid)}")
    for f, twin in duplicates.items():
        print(f"  {f}  (r = {scaled['train'][f].corr(scaled['train'][twin]):+.3f} with {twin})")
    print()
selected = (list(mrmr_classif(X=scaled["train"][candidates], y=pd.Series(train["label"].astype(int)),
                              K=min(N_FEATURES, len(candidates)), show_progress=False))
            if N_FEATURES > 0 else [])                # N_FEATURES = 0: TI-RADS features only
selected = clinical + selected                        # TI-RADS features first, then mRMR order
print(f"{len(radiomic)} usable radiomics features, {len(candidates)} after the filter; mRMR selected {N_FEATURES}"
      + (f", plus {len(clinical)} TI-RADS features" if clinical else "") + ":")
for i, f in enumerate(selected, 1):
    print(f"  {i:2d}. {f}")

Near-duplicates dropped (|r| > 0.95 on train): 48 of 109
  original_shape2D_MaximumDiameterRelative  (r = +0.997 with original_shape2D_MajorAxisLengthRelative)
  original_shape2D_PerimeterRelative  (r = +0.986 with original_shape2D_MajorAxisLengthRelative)
  original_shape2D_MeshSurfaceRelative  (r = +1.000 with original_shape2D_PixelSurfaceRelative)
  original_shape2D_MinorAxisLengthRelative  (r = +0.956 with original_shape2D_PixelSurfaceRelative)
  original_gldm_SmallDependenceEmphasis  (r = -0.993 with original_glcm_Idm)
  original_glcm_Id  (r = +0.999 with original_glcm_Idm)
  original_glszm_ZonePercentage  (r = -0.971 with original_glcm_Idm)
  original_glrlm_RunPercentage  (r = -0.971 with original_glcm_Idm)
  original_gldm_LargeDependenceEmphasis  (r = +0.958 with original_glcm_Idm)
  original_firstorder_Maximum  (r = +0.959 with original_firstorder_Range)
  original_glrlm_RunLengthNonUniformityNormalized  (r = +1.000 with original_glszm_SizeZoneNonUniformityNormalized)
  origina

## C3. Prompts

Same format as the main notebook when `VALUE_FORMAT = "decimal"`. With `"zscore"` or `"percentile"` each value states its reference (the training nodules); percentiles are computed against the train split only.

`"rich"` adds a plain-language band (much lower / lower / close to / higher / much higher than typical, from train-split percentiles 10/30/70/90) and, away from typical, what the value means for the nodule (`conformal_triage/semantics.py`). The phrases say what a feature measures, never which direction is malignant. `"semantic_only"` keeps the band and meaning but drops the numbers.

In [14]:
TRAIN_REF = np.sort(scaled["train"][selected].to_numpy(dtype=np.float64), axis=0)

def percentiles(X):
    """Percentile (0-100) of each value among the train-split nodules."""
    return np.stack([np.searchsorted(TRAIN_REF[:, j], X[:, j], side="right") / len(TRAIN_REF) * 100
                     for j in range(X.shape[1])], axis=1)

def prompt_values(X):
    """Values as written in the prompt: percentiles for "percentile", z-scores otherwise."""
    return percentiles(X) if VALUE_FORMAT == "percentile" else X

data = {}
for s in SPLITS:
    X = scaled[s][selected].to_numpy(dtype=np.float32)
    meta = tables[s][["id", "official_split", "label"]].reset_index(drop=True)
    data[s] = pd.DataFrame({
        "sample_id": meta["official_split"].astype(str) + "/" + meta["id"].astype(str),
        "label": meta["label"].astype(int),
        "prompt": [ct.render_prompt(selected, x, value_format=VALUE_FORMAT, percentiles=p) for x, p in zip(prompt_values(X), percentiles(X))],
    })

print("Example prompt (validation, first nodule):\n")
print(data["validation"]["prompt"].iloc[0])

Example prompt (validation, first nodule):

Classify this thyroid nodule as benign or malignant based on the following standardized radiomic features. The 'Taller Than Wide Ratio' feature is measured at 1.358. The 'Echogenicity Ratio' feature is measured at 0.164. The 'Solid Echogenicity Ratio' feature is measured at -0.093. The 'Punctate Foci Density' feature is measured at -0.301. The 'Solidity' feature is measured at 0.384. The 'Margin Sharpness' feature is measured at -1.203. The 'Anechoic Fraction' feature is measured at -0.425. The 'Major Axis Length Relative' feature is measured at 1.797. The 'Small Area Low Gray Level Emphasis' feature is measured at -0.336. The 'Elongation' feature is measured at 1.913. The 'Pixel Surface Relative' feature is measured at 3.579. The 'Range' feature is measured at 1.085. The 'Perimeter Surface Ratio Relative' feature is measured at -1.127. The 'Dependence Variance' feature is measured at 0.249. The 'Mean Absolute Deviation' feature is measured a

## C4. Load the backbone in 4-bit and add LoRA

LoRA rank 16 on the attention Q/V projections, as in the main notebook. Architectures with a fused QKV projection (e.g. Phi) use `qkv_proj`.

In [15]:
assert torch.cuda.is_available(), "Part C needs a GPU runtime: Runtime -> Change runtime type -> L4"

# Gated models (e.g. Llama) need a Hugging Face token saved in Colab Secrets as HF_TOKEN.
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    print("Logged in to Hugging Face with HF_TOKEN from Colab Secrets")
except Exception as e:
    print(f"No HF_TOKEN available ({type(e).__name__}); only public models can be downloaded")

from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, device_map="auto", torch_dtype=DTYPE,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=DTYPE))
model.config.pad_token_id = tokenizer.pad_token_id
model = prepare_model_for_kbit_training(model)

module_names = {n.split(".")[-1] for n, _ in model.named_modules()}
ALL_LINEAR = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
if LORA_TARGETS == "all":
    TARGETS = [m for m in ALL_LINEAR if m in module_names]
else:
    TARGETS = ["q_proj", "v_proj"] if {"q_proj", "v_proj"} <= module_names else ["qkv_proj"]
model = get_peft_model(model, LoraConfig(
    task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=TARGETS))
print("LoRA targets:", TARGETS)
model.print_trainable_parameters()
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")

Logged in to Hugging Face with HF_TOKEN from Colab Secrets


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


LoRA targets: ['q_proj', 'v_proj']
trainable params: 2,182,144 || all params: 1,545,899,520 || trainable%: 0.1412
VRAM allocated: 1.52 GiB


## C5. Tokenized datasets

In [16]:
from torch.utils.data import Dataset

class PromptDataset(Dataset):
    def __init__(self, df):
        self.texts, self.labels = df["prompt"].tolist(), df["label"].tolist()
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        enc = tokenizer(self.texts[i], truncation=True, max_length=MAX_LENGTH,
                        padding="max_length", return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "labels": torch.tensor(self.labels[i], dtype=torch.long)}

longest = max(len(tokenizer(p)["input_ids"]) for s in SPLITS for p in data[s]["prompt"])
print(f"Longest prompt: {longest} tokens (MAX_LENGTH = {MAX_LENGTH})")
assert longest <= MAX_LENGTH, f"Prompts would be truncated: set MAX_LENGTH above {longest}"
train_ds, val_ds = PromptDataset(data["train"]), PromptDataset(data["validation"])

Longest prompt: 442 tokens (MAX_LENGTH = 512)


## C6. Fine-tune

Class-weighted cross-entropy, cosine schedule with 5% warmup, the best epoch chosen by validation AUC. `model_accepts_loss_kwargs = False` makes the Trainer divide the loss by the accumulation steps (the gradient-accumulation fix from main notebook B5). The table below updates after each epoch.

In [17]:
from transformers import Trainer, TrainingArguments

counts = data["train"]["label"].value_counts().sort_index()
class_weights = torch.tensor((counts.sum() / (2.0 * counts)).values, dtype=torch.float32)
print("class weights [benign, malignant]:", class_weights.tolist())

class WeightedTrainer(Trainer):
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.model_accepts_loss_kwargs = False

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=class_weights.to(out.logits.device))(out.logits, labels)
        return (loss, out) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    p = torch.softmax(torch.tensor(np.asarray(logits, dtype=np.float32)), dim=-1).numpy()[:, 1]
    pred = (p >= 0.5).astype(int)
    return {"auc": roc_auc_score(labels, p), "acc": accuracy_score(labels, pred),
            "sensitivity": recall_score(labels, pred)}

steps_per_epoch = math.ceil(len(train_ds) / (BATCH * GRAD_ACCUM))
trainer = WeightedTrainer(
    model=model,
    args=TrainingArguments(
        output_dir=str(RUN_DIR / "checkpoints"), num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH * 2,
        gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR,
        lr_scheduler_type="cosine", warmup_steps=math.ceil(0.05 * steps_per_epoch * EPOCHS),
        logging_steps=20, save_strategy="epoch", eval_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="auc",
        bf16=PRECISION_USED == "bf16", fp16=PRECISION_USED == "fp16",
        gradient_checkpointing=True, report_to="none", seed=SEED),
    train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)

start = time.time()
trainer.train()
TRAIN_MINUTES = round((time.time() - start) / 60, 1)
trainer.save_model(str(RUN_DIR / "adapter"))
pd.DataFrame(trainer.state.log_history).to_csv(RUN_DIR / "train_log.csv", index=False)
print(f"Training took {TRAIN_MINUTES} min; adapter saved to {RUN_DIR / 'adapter'}")

class weights [benign, malignant]: [0.764598548412323, 1.4448275566101074]


Epoch,Training Loss,Validation Loss,Auc,Acc,Sensitivity
1,0.682697,0.755669,0.762228,0.662844,0.059603
2,0.661265,0.649045,0.772999,0.685780,0.139073
3,0.661180,0.633201,0.785663,0.708716,0.304636
4,0.558905,0.576555,0.783664,0.727064,0.516556
5,0.538238,0.555549,0.784257,0.722477,0.668874


Training took 37.1 min; adapter saved to /content/drive/MyDrive/thesis_model_sweep/Qwen2.5-1.5B_f16_e5_s42_pooled_seed42_v2_corr95_clin7/adapter


## C7. Validation metrics per epoch

In [18]:
log = pd.DataFrame(trainer.state.log_history)
per_epoch = log.dropna(subset=["eval_auc"])[["epoch", "eval_loss", "eval_auc", "eval_acc", "eval_sensitivity"]]
display(per_epoch.round(4).reset_index(drop=True))
print("Best validation AUC:", round(per_epoch["eval_auc"].max(), 4))

,epoch,eval_loss,eval_auc,eval_acc,eval_sensitivity
0,1.0,0.7557,0.7622,0.6628,0.0596
1,2.0,0.6490,0.7730,0.6858,0.1391
2,3.0,0.6332,0.7857,0.7087,0.3046
3,4.0,0.5766,0.7837,0.7271,0.5166
4,5.0,0.5555,0.7843,0.7225,0.6689


Best validation AUC: 0.7857


## C8. Predict P(malignant) on validation, calibration and test

In [19]:
# The Trainer wraps forward() in a bf16 autocast that stays on after training. Under it the two
# classifier logits are rounded to bf16 (steps of 1/64), so many nodules get exactly the same
# P(malignant). Removing the wrapper gives full-precision scores for the threshold analysis.
model = trainer.accelerator.unwrap_model(trainer.model, keep_fp32_wrapper=False)
probs, labels = {}, {}
for s in ["validation", "calibration", "test"]:
    probs[s] = ct.predict_malignant_proba(model, tokenizer, data[s]["prompt"].tolist(),
                                          batch_size=BATCH * 2, max_length=MAX_LENGTH)
    labels[s] = data[s]["label"].to_numpy()
    data[s].assign(prob_malignant=probs[s]).drop(columns="prompt").to_csv(RUN_DIR / f"probs_{s}.csv", index=False)
print("Saved per-nodule probabilities to", RUN_DIR)
print(f"distinct P(malignant) values on calibration: {len(np.unique(probs['calibration']))} of {len(probs['calibration'])}")

Saved per-nodule probabilities to /content/drive/MyDrive/thesis_model_sweep/Qwen2.5-1.5B_f16_e5_s42_pooled_seed42_v2_corr95_clin7
distinct P(malignant) values on calibration: 524 of 524


## C9. Classification metrics

`auc_correct_softmax` measures how well softmax confidence (distance of P from 0.5) separates correct from incorrect predictions; 0.5 is chance. This is report Section 10.2.

In [20]:
cls = {}
for s in ["validation", "calibration", "test"]:
    p, y = probs[s], labels[s]
    pred = (p >= 0.5).astype(int)
    cls[s] = {"acc": accuracy_score(y, pred), "f1": f1_score(y, pred), "auc": roc_auc_score(y, p),
              "sens": recall_score(y, pred), "auc_correct_softmax": roc_auc_score(pred == y, np.abs(p - 0.5))}
cls = pd.DataFrame(cls).T
display(cls.round(3))

,acc,f1,auc,sens,auc_correct_softmax
validation,0.706,0.407,0.786,0.291,0.737
calibration,0.758,0.517,0.802,0.374,0.680
test,0.763,0.532,0.781,0.388,0.645


## C10. LTT on the softmax score (calibration → test)

Same rule and candidate thresholds as main notebook D6: P ≤ t is auto-benign (R1: share of cancers ≤ `ALPHA_BENIGN`), P ≥ u is auto-malignant (R2: share of benign ≤ `ALPHA_MALIGNANT`). Thresholds are tested from the safest candidate onwards, stopping at the first failure. `certified = False` everywhere means nothing could be guaranteed.

In [21]:
P_T_GRID = [round(x, 2) for x in np.arange(0.05, 0.50, 0.05)]
P_U_GRID = [round(x, 2) for x in np.arange(0.95, 0.50, -0.05)]
pc, yc, pt, yt = probs["calibration"], labels["calibration"], probs["test"], labels["test"]

rule = ct.calibrate_triage(pc, yc, P_T_GRID, P_U_GRID, ALPHA_BENIGN, ALPHA_MALIGNANT, DELTA)
print(f"R1 (auto-benign), alpha = {ALPHA_BENIGN}:"); display(rule.benign_table.round(4))
print(f"R2 (auto-malignant), alpha = {ALPHA_MALIGNANT}:"); display(rule.malignant_table.round(4))
print(f"Locked rule: t = {rule.t}, u = {rule.u}")
display(ct.triage_report(rule.decide(pt), yt).to_frame("test").round(3))

R1 (auto-benign), alpha = 0.05:


,threshold,n,errors,observed_rate,p_value,certified
0,0.05,30,1,0.0333,0.5535,False
1,0.10,114,9,0.0789,0.9401,False
2,0.15,181,21,0.1160,0.9999,False
3,0.20,225,34,0.1511,1.0000,False
4,0.25,274,45,0.1642,1.0000,False
5,0.30,322,58,0.1801,1.0000,False
6,0.35,354,72,0.2034,1.0000,False
7,0.40,389,84,0.2159,1.0000,False
8,0.45,419,100,0.2387,1.0000,False


R2 (auto-malignant), alpha = 0.2:


,threshold,n,errors,observed_rate,p_value,certified
0,0.95,0,0,NaN,1.0000,False
1,0.90,0,0,NaN,1.0000,False
2,0.85,2,1,0.5000,0.9600,False
3,0.80,8,1,0.1250,0.5033,False
4,0.75,16,2,0.1250,0.3518,False
5,0.70,28,2,0.0714,0.0612,False
6,0.65,31,2,0.0645,0.0374,False
7,0.60,41,2,0.0488,0.0066,False
8,0.55,56,5,0.0893,0.0214,False


Locked rule: t = None, u = None


,test
n,438.0
auto_benign,0.0
auto_benign_malignant,0.0
auto_benign_malignant_rate,NaN
auto_malignant,0.0
auto_malignant_benign,0.0
auto_malignant_benign_rate,NaN
referred,438.0
automation_rate,0.0
cancers_auto_benign_share,0.0


## C11. Miss-rate guarantee (exploratory, report Section 10.3)

Bounds the share of **all** cancers sent to auto-benign (P ≤ t). Candidates are fixed in advance (0.01 to 0.50). Calibration and test are not exchangeable (report Section 5), so the realised test miss rate can exceed the target.

In [22]:
MISS_GRID = [round(x, 2) for x in np.arange(0.01, 0.51, 0.01)]
n_cancer = int((yc == 1).sum())

miss = {}
for alpha in (0.05, 0.10):
    t = None
    for cand in MISS_GRID:
        k = int(((pc <= cand) & (yc == 1)).sum())
        if binom.cdf(k, n_cancer, alpha) > DELTA:
            break
        t = cand
    miss[f"miss rate <= {alpha:.0%}"] = {
        "threshold t": t,
        "calibration automation": (pc <= t).mean() if t is not None else np.nan,
        "test automation": (pt <= t).mean() if t is not None else np.nan,
        "test miss rate": ((pt <= t) & (yt == 1)).sum() / (yt == 1).sum() if t is not None else np.nan,
    }
miss = pd.DataFrame(miss).T
display(miss.round(3))

,threshold t,calibration automation,test automation,test miss rate
miss rate <= 5%,0.07,0.128,0.148,0.059
miss rate <= 10%,0.11,0.246,0.276,0.099


## C12. Split conformal prediction (marginal and Mondrian)

The classical alternative to LTT: calibrate a threshold so that the true label is in the prediction set for at least $1-\alpha$ of the nodules, and refer every nodule whose set holds both labels. *Marginal* uses one threshold for both labels; *Mondrian* calibrates each label on its own nodules, here with $\alpha = 0.05$ for cancers, so at most about 5% of cancers get the single label "benign" (a guarantee on missed cancers, like the miss-rate rule). Neither bounds the share of cancers *among* the single-label "benign" sets, which is what LTT's $R_1$ asks for; the last columns show that share on test.

In [23]:
CP_SETTINGS = {"marginal, alpha 0.10": ("marginal", 0.10),
               "marginal, alpha 0.05": ("marginal", 0.05),
               "Mondrian, alpha 0.10 benign / 0.05 malignant": ("mondrian", (0.10, 0.05))}
sc_cal, sc_test = ct.softmax_scores(probs["calibration"]), ct.softmax_scores(probs["test"])
cp = {}
for name, (kind, alpha) in CP_SETTINGS.items():
    calibrate = ct.calibrate_marginal if kind == "marginal" else ct.calibrate_mondrian
    q_hat = calibrate(sc_cal, labels["calibration"], alpha)
    cp[name] = ct.set_report(ct.prediction_sets(sc_test, q_hat), labels["test"])
cp = pd.DataFrame(cp).T
print("Test set (thresholds from calibration):")
display(cp[["coverage", "coverage_benign", "coverage_malignant", "benign_only_rate", "cancer_share_in_benign_only",
            "malignant_only_rate", "benign_share_in_malignant_only", "both_labels_rate"]].round(3))

Test set (thresholds from calibration):


,coverage,coverage_benign,coverage_malignant,benign_only_rate,cancer_share_in_benign_only,malignant_only_rate,benign_share_in_malignant_only,both_labels_rate
"marginal, alpha 0.10",0.881,1.000,0.658,0.603,0.197,0.041,0.000,0.356
"marginal, alpha 0.05",0.941,1.000,0.829,0.406,0.146,0.007,0.000,0.587
"Mondrian, alpha 0.10 benign / 0.05 malignant",0.909,0.906,0.914,0.247,0.120,0.224,0.276,0.530


## C13. Save the run and compare with earlier runs

In [24]:
row = {"run": RUN_TAG, "model": MODEL_NAME, "feature_set": FEATURE_SET, "value_format": VALUE_FORMAT, "max_corr": MAX_CORR, "clinical": CLINICAL_FEATURES, "split_scheme": SPLIT_SCHEME, "n_features": len(selected), "epochs": EPOCHS, "lr": LR, "lora": LORA_TARGETS,
       "gpu": GPU, "precision": PRECISION_USED, "minutes": TRAIN_MINUTES}
for s in cls.index:
    for m in cls.columns:
        row[f"{s}_{m}"] = cls.loc[s, m]
row.update({"ltt_t": rule.t, "ltt_u": rule.u,
            "ltt_test_auto_benign": int((rule.decide(pt) == 0).sum()),
            "ltt_test_auto_malignant": int((rule.decide(pt) == 1).sum())})
for alpha, tag in ((0.05, "miss5"), (0.10, "miss10")):
    r = miss.loc[f"miss rate <= {alpha:.0%}"]
    row.update({f"{tag}_t": r["threshold t"], f"{tag}_cal_automation": r["calibration automation"],
                f"{tag}_test_automation": r["test automation"], f"{tag}_test_miss_rate": r["test miss rate"]})

m = cp.loc["Mondrian, alpha 0.10 benign / 0.05 malignant"]
row.update({"cp_mondrian_coverage_malignant": m["coverage_malignant"], "cp_mondrian_benign_only": m["benign_only_rate"],
            "cp_mondrian_cancer_share_benign_only": m["cancer_share_in_benign_only"]})

(RUN_DIR / "config.json").write_text(json.dumps(
    {"model": MODEL_NAME, "feature_set": FEATURE_SET, "value_format": VALUE_FORMAT, "max_corr": MAX_CORR, "clinical": CLINICAL_FEATURES, "split_scheme": SPLIT_SCHEME, "n_features": N_FEATURES, "epochs": EPOCHS, "lr": LR, "batch": BATCH,
     "grad_accum": GRAD_ACCUM, "max_length": MAX_LENGTH, "seed": SEED, "precision": PRECISION_USED,
     "gpu": GPU, "lora_targets": TARGETS, "selected_features": selected}, indent=2))
pd.Series(row).to_json(RUN_DIR / "metrics.json", indent=2)
summary_path = Path(OUT_DIR) / "summary.csv"
summary = pd.DataFrame([row])
if summary_path.exists():   # keep earlier rows even if the columns changed
    summary = pd.concat([pd.read_csv(summary_path), summary], ignore_index=True)
summary.to_csv(summary_path, index=False)

cols = ["run", "feature_set", "value_format", "max_corr", "lr", "lora", "split_scheme", "gpu", "precision", "minutes", "validation_auc", "calibration_auc", "test_auc",
        "calibration_auc_correct_softmax", "ltt_t", "ltt_u", "miss5_t", "miss5_test_miss_rate"]
print("All runs so far:")
display(summary[[c for c in cols if c in summary.columns]].round(3))
print("Saved:", RUN_DIR, "and", summary_path)

All runs so far:


,run,feature_set,value_format,max_corr,lr,lora,split_scheme,gpu,precision,minutes,validation_auc,calibration_auc,test_auc,calibration_auc_correct_softmax,ltt_t,ltt_u,miss5_t,miss5_test_miss_rate
0,Qwen2.5-1.5B_f8_e5_s42,NaN,NaN,NaN,NaN,NaN,NaN,NVIDIA L4,bf16,16.2,0.697,0.714,0.667,0.658,NaN,NaN,0.22,0.081
1,Qwen2.5-7B_f8_e5_s42,NaN,NaN,NaN,NaN,NaN,NaN,NVIDIA L4,bf16,57.3,0.697,0.686,0.669,0.448,NaN,NaN,0.28,0.030
2,Qwen2.5-7B_f16_e5_s42,NaN,NaN,NaN,NaN,NaN,NaN,NVIDIA L4,bf16,85.3,0.702,0.715,0.666,0.660,NaN,NaN,0.19,0.034
3,Qwen2.5-1.5B_f16_e5_s42_pooled_seed42_v2,v2,decimal,NaN,NaN,NaN,pooled_seed42,NVIDIA L4,bf16,26.1,0.753,0.757,0.750,0.700,NaN,NaN,0.16,0.053
4,Qwen2.5-1.5B_f16_e5_s42_pooled_seed42_corr95,v1,decimal,0.95,NaN,NaN,pooled_seed42,NVIDIA L4,bf16,26.6,0.709,0.705,0.699,0.684,NaN,NaN,0.22,0.026
5,Qwen2.5-7B_f16_e5_s42_pooled_seed42_v2_corr95,v2,decimal,0.95,NaN,NaN,pooled_seed42,NVIDIA L4,bf16,90.2,0.708,0.705,0.713,0.702,NaN,NaN,0.09,0.026
6,Qwen2.5-1.5B_f16_e5_s42_pooled_seed42_v2_corr95,v2,decimal,0.95,NaN,NaN,pooled_seed42,NVIDIA L4,bf16,26.4,0.750,0.752,0.754,0.692,NaN,NaN,0.16,0.086
7,Qwen2.5-1.5B_f16_e5_s42_pooled_seed42_v2_zscor...,v2,zscore,0.95,NaN,NaN,pooled_seed42,NVIDIA L4,bf16,36.9,0.729,0.756,0.759,0.721,NaN,NaN,0.21,0.059
8,Qwen2.5-1.5B_f16_e5_s42_pooled_seed42_v2_perce...,v2,percentile,0.95,NaN,NaN,pooled_seed42,NVIDIA L4,bf16,36.5,0.709,0.713,0.734,0.617,NaN,NaN,0.26,0.033
9,Llama-2-7b-hf_f16_e5_s42_pooled_seed42_v2_corr...,v2,decimal,0.95,0.0,all,pooled_seed42,NVIDIA L4,bf16,182.4,0.705,0.709,0.731,0.696,NaN,NaN,0.17,0.007


Saved: /content/drive/MyDrive/thesis_model_sweep/Qwen2.5-1.5B_f16_e5_s42_pooled_seed42_v2_corr95_clin7 and /content/drive/MyDrive/thesis_model_sweep/summary.csv


## Done

- Results are in `OUT_DIR` on Google Drive. Send `summary.csv` (and the run folders if needed) for the comparison.
- Next model: change `MODEL_NAME` in the Configuration cell, then **Restart session** and **Run all**.
- Finished for now: Runtime → **Disconnect and delete runtime**.